# Evaporating Universe — Paper I
## NB06: Model Selection (AIC / BIC / Δχ²)

**EU vs ΛCDM — Information Criteria**

| Comparison | EU source | ΛCDM source | Expected |
|:-----------|:----------|:------------|:---------|
| Baseline (CMB+BAO+SNe) | NB05 C2 | Minimizer | Δχ² ≈ 0 |
| Extended (+SH0ES+DES) | NB05 C2 + proxy | Minimizer + proxy | Δχ² ≈ −50 |

> **Key advantage:** k_EU = k_ΛCDM = 7 → ΔAIC = ΔBIC = Δχ² (pure goodness-of-fit)


---


In [ ]:
# ============================================================
# §1. SETUP
# ============================================================
import os, sys, json, time
import numpy as np

# -- Install Cobaya --
try:
    import cobaya
    print(f'[OK] Cobaya {cobaya.__version__} already installed')
except ImportError:
    print('[INSTALL] Installing Cobaya...')
    os.system('pip install -q cobaya')
    import cobaya
    print(f'[OK] Cobaya {cobaya.__version__} installed')

# -- Install cosmological packages (CLASS + likelihoods) --
PACKAGES_PATH = '/content/packages'
if not os.path.exists(os.path.join(PACKAGES_PATH, 'code', 'classy')):
    print('[INSTALL] Downloading CLASS + likelihoods (~5 min)...')
    os.system(f'cobaya-install cosmo -p {PACKAGES_PATH} --no-progress')
    print('[OK] Cosmological packages installed')
else:
    print(f'[OK] Packages already at {PACKAGES_PATH}')

# -- Install Planck clik/clipy (needed for lensing likelihood) --
if not os.path.exists(os.path.join(PACKAGES_PATH, 'code', 'planck', 'clipy')):
    print('[INSTALL] Compiling Planck clipy (~2 min)...')
    os.system(f'cobaya-install planck_2018_lensing.clik -p {PACKAGES_PATH} --no-progress')
    print('[OK] Planck clipy installed')
else:
    print(f'[OK] Planck clipy already compiled')

# -- Upload EU results --
from google.colab import files as _files

REQUIRED = ['NB05_C2_results.json', 'NB05_D2_results.json', 'NB03_B_results.json']
OPTIONAL = ['NB06_lcdm_minimizer.json']  # Upload to skip BOBYQA (~35 min)

print('\nRequired uploads:')
for f in REQUIRED:
    print(f'  • {f}')
print('Optional (skip minimizer):')
for f in OPTIONAL:
    print(f'  • {f}')

all_expected = REQUIRED + OPTIONAL
missing = [f for f in all_expected if not os.path.exists(f'/content/{f}')]
if missing:
    print(f'\nUpload now: {missing}')
    uploaded = _files.upload()
    for name, content in uploaded.items():
        with open(f'/content/{name}', 'wb') as f:
            f.write(content)
        print(f'  [OK] {name}')

# Verify REQUIRED (no fallback)
for f in REQUIRED:
    assert os.path.exists(f'/content/{f}'), f'FATAL: {f} is required. No fallback.'

# -- Detect minimizer cache --
MINIMIZER_CACHE = '/content/NB06_lcdm_minimizer.json'
USE_CACHE = os.path.exists(MINIMIZER_CACHE)

if USE_CACHE:
    with open(MINIMIZER_CACHE) as f:
        _cache = json.load(f)
    print(f'\n[CACHE] LCDM minimizer loaded — BOBYQA will be SKIPPED')
    print(f'        Source: {_cache["metadata"]["date"]}')
    print(f'        Method: {_cache["metadata"]["method"]}')
else:
    print(f'\n[INFO] No minimizer cache — BOBYQA will run (~35 min)')

# -- Load EU results --
with open('/content/NB05_C2_results.json') as f:
    c2 = json.load(f)
print(f'\n[OK] C2 loaded: chi2 = {c2["chi2_bestfit"]["total"]:.1f}')

with open('/content/NB05_D2_results.json') as f:
    d2 = json.load(f)
print(f'[OK] D2 loaded: chi2 = {d2["chi2_bestfit"]["total"]:.1f}')
print(f'     D2 status: {d2["_metadata"].get("status", "N/A")}')

with open('/content/NB03_B_results.json') as f:
    nb03b = json.load(f)
print(f'[OK] NB03-B loaded: S8_eu_pred = {nb03b["sigma8"]["S8_eu"]:.4f}')

print('\n' + '=' * 60)
print('SETUP COMPLETE')
print('  C2 (Baseline): CMB + BAO + SNe')
print('  NB03-B: CLASS-EU S8 prediction (Mode B)')
print('  D2 (Extended): CMB + BAO + SNe + SH0ES + DES-Y3')
if USE_CACHE:
    print('  LCDM Minimizer: CACHED (BOBYQA skipped)')
else:
    print('  LCDM Minimizer: WILL RUN (BOBYQA, best_of=4)')
print('=' * 60)


---


In [ ]:
# ============================================================
# §2. LCDM MINIMIZER - Baseline (CMB + BAO + SNe)
# ============================================================
# If NB06_lcdm_minimizer.json was uploaded, skip BOBYQA.
# Otherwise, run the full minimizer (~30 min).
# ============================================================

print('=' * 60)
print('LCDM MINIMIZER - Baseline')
print('=' * 60)

if USE_CACHE:
    # ── Load from cache ──
    chi2_lcdm_base  = _cache['baseline']['chi2_total']
    H0_lcdm_bf      = _cache['baseline']['H0']
    sigma8_lcdm_bf  = _cache['baseline']['sigma8']
    S8_lcdm_bf      = _cache['baseline']['S8']
    Omega_m_lcdm_bf = _cache['baseline']['Omega_m']
    rdrag_lcdm_bf   = _cache['baseline']['rdrag']
    lcdm_chi2_base  = _cache['baseline']['per_likelihood']
    print(f'\n[CACHE] Baseline loaded from NB06_lcdm_minimizer.json')
    print(f'  chi2_total  = {chi2_lcdm_base:.3f}')
    print(f'  H0          = {H0_lcdm_bf:.3f} km/s/Mpc')
    print(f'  sigma8      = {sigma8_lcdm_bf:.4f}')
    print(f'  S8          = {S8_lcdm_bf:.4f}')
    print(f'  Omega_m     = {Omega_m_lcdm_bf:.4f}')
    print(f'  r_d         = {rdrag_lcdm_bf:.2f} Mpc')

else:
    # ── Run BOBYQA ──
    from cobaya.run import run as cobaya_run
    import copy

    info_baseline = {
        'likelihood': {
            'planck_NPIPE_highl_CamSpec.TTTEEE': None,
            'planck_2018_lowl.TT': None,
            'planck_2018_lowl.EE': None,
            'planck_2018_lensing.clik': None,
            'bao.desi_dr2': None,
            'sn.pantheonplus': None,
        },
        'theory': {
            'classy': {
                'extra_args': {
                    'N_ur': 2.0328,
                    'N_ncdm': 1,
                    'm_ncdm': 0.0589,
                    'non_linear': 'halofit',
                    'P_k_max_1/Mpc': 10,
                }
            }
        },
        'params': {
            'omega_b':     {'prior': {'min': 0.019, 'max': 0.025},
                            'ref': 0.02237, 'proposal': 0.0001},
            'omega_cdm':   {'prior': {'min': 0.08,  'max': 0.16},
                            'ref': 0.1200,  'proposal': 0.0005},
            'theta_s_100': {'prior': {'min': 1.03,  'max': 1.05},
                            'ref': 1.04092, 'proposal': 0.0002},
            'tau_reio':    {'prior': {'min': 0.01,  'max': 0.12},
                            'ref': 0.0544,  'proposal': 0.005},
            'logA':        {'prior': {'min': 2.5,   'max': 3.5},
                            'ref': 3.044,   'proposal': 0.01,
                            'drop': True},
            'A_s':         {'value': 'lambda logA: 1e-10*np.exp(logA)'},
            'n_s':         {'prior': {'min': 0.9,   'max': 1.05},
                            'ref': 0.9649,  'proposal': 0.003},
            'A_planck':    {'prior': {'dist': 'norm', 'loc': 1, 'scale': 0.0025},
                            'ref': 1.0,     'proposal': 0.0005},
            'H0':       {'derived': True},
            'sigma8':   {'derived': True},
            'Omega_m':  {'derived': True},
            'rdrag':    {'derived': True},
            'S8': {'derived': 'lambda sigma8, Omega_m: sigma8 * (Omega_m / 0.3)**0.5'},
        },
        'sampler': {
            'minimize': {
                'method': 'bobyqa',
                'best_of': 4,
            }
        },
        'force': True,
        'packages_path': '/content/packages',
        'output': '/content/lcdm_baseline',
    }

    print('Running LCDM minimizer (Baseline)...')
    print(f'  Likelihoods: {list(info_baseline["likelihood"].keys())}')
    print(f'  Method: BOBYQA, best_of=4')
    print()

    t0 = time.time()
    updated_info_base, sampler_base = cobaya_run(info_baseline)
    dt_base = time.time() - t0

    # Load results from disk (Cobaya version compatibility)
    with open('/content/lcdm_baseline.minimum.txt') as _f:
        _hdr = _f.readline().strip().lstrip('#').split()
    _raw = np.loadtxt('/content/lcdm_baseline.minimum.txt')
    if _raw.ndim == 1:
        _raw = _raw.reshape(1, -1)
    _best = _raw[np.argmin(_raw[:, _hdr.index('minuslogpost')])]
    
    class _DiskBF:
        def __init__(self, hdr, row):
            self._h, self._r = hdr, row
            self.columns = hdr
        def __getitem__(self, key):
            return float(self._r[self._h.index(key)])
    bf_base = _DiskBF(_hdr, _best)

    _chi2_cols = [c for c in bf_base.columns if c.startswith('chi2__')]
    if _chi2_cols:
        chi2_lcdm_base = sum(bf_base[c] for c in _chi2_cols)
    else:
        _ll = [c for c in bf_base.columns if c.startswith('minuslogl__')]
        chi2_lcdm_base = 2 * sum(bf_base[c] for c in _ll)

    H0_lcdm_bf      = bf_base['H0']
    sigma8_lcdm_bf  = bf_base['sigma8']
    Omega_m_lcdm_bf = bf_base['Omega_m']
    S8_lcdm_bf      = bf_base['S8']
    rdrag_lcdm_bf   = bf_base['rdrag']

    # Per-likelihood chi2 for breakdown
    lcdm_chi2_base = {}
    for col in bf_base.columns:
        if col.startswith('chi2__'):
            lcdm_chi2_base[col.replace('chi2__', '')] = float(bf_base[col])
        elif col.startswith('minuslogl__'):
            lcdm_chi2_base[col.replace('minuslogl__', '')] = 2 * float(bf_base[col])

    print(f'\n{"="*60}')
    print(f'LCDM Baseline Best-Fit (took {dt_base:.0f}s)')
    print(f'{"="*60}')
    print(f'  chi2_total  = {chi2_lcdm_base:.3f}')
    print(f'  H0          = {H0_lcdm_bf:.3f} km/s/Mpc')
    print(f'  sigma8      = {sigma8_lcdm_bf:.4f}')
    print(f'  S8          = {S8_lcdm_bf:.4f}')
    print(f'  Omega_m     = {Omega_m_lcdm_bf:.4f}')
    print(f'  r_d         = {rdrag_lcdm_bf:.2f} Mpc')


---


In [ ]:
# ============================================================
# §3. EXTENDED COMPARISON - Add SH0ES (prediction penalty)
# ============================================================
# Both models predict H0 from baseline (CMB+BAO+SNe),
# then SH0ES penalty is added as post-hoc check.
#
# DES-Y3 S8 (0.776) is NOT included: it was derived under
# LCDM assumptions (kernel bias). EU vs DES -> NB08.
# ============================================================

print('=' * 60)
print('EXTENDED COMPARISON (+SH0ES)')
print('=' * 60)

H0_SHOES = 73.17
H0_SHOES_ERR = 0.86
print(f'  SH0ES: H0 = {H0_SHOES} +/- {H0_SHOES_ERR} km/s/Mpc (Breuval+ 2024)')

# -- LCDM: SH0ES penalty at baseline best-fit --
chi2_eu_base = c2['chi2_bestfit']['total']

chi2_shoes_lcdm = ((H0_lcdm_bf - H0_SHOES) / H0_SHOES_ERR)**2
chi2_lcdm_ext = chi2_lcdm_base + chi2_shoes_lcdm
print(f'\n  LCDM H0 = {H0_lcdm_bf:.2f} -> chi2_SH0ES = {chi2_shoes_lcdm:.1f}')
print(f'  chi2_lcdm_ext = {chi2_lcdm_base:.1f} + {chi2_shoes_lcdm:.1f} = {chi2_lcdm_ext:.1f}')

# -- EU: SH0ES penalty at C2 H0_LKI prediction --
H0_LKI_eu_base = c2['derived_params']['H0_LKI']['mean']
chi2_shoes_eu = ((H0_LKI_eu_base - H0_SHOES) / H0_SHOES_ERR)**2
chi2_eu_ext = chi2_eu_base + chi2_shoes_eu
print(f'\n  EU H0_LKI (C2) = {H0_LKI_eu_base:.2f} -> chi2_SH0ES = {chi2_shoes_eu:.2f}')
print(f'  chi2_eu_ext = {chi2_eu_base:.1f} + {chi2_shoes_eu:.2f} = {chi2_eu_ext:.1f}')

# -- D2 reference (EU fit with SH0ES+DES likelihoods) --
H0_LKI_eu_ext = d2['derived_params']['H0_LKI']['mean']
S8_eu_ext     = d2['derived_params']['S8']['mean']
print(f'\n  [REF] D2 (full fit): H0_LKI={H0_LKI_eu_ext:.2f}, S8={S8_eu_ext:.4f}')

# -- S8 internal consistency (for §6 tension summary) --
S8_eu_pred    = nb03b['sigma8']['S8_eu']       # CLASS-EU prediction
S8_eu_d2_mean = d2['derived_params']['S8']['mean']
S8_eu_d2_std  = d2['derived_params']['S8']['std']
print(f'  [REF] S8 internal: pred={S8_eu_pred:.4f} vs D2={S8_eu_d2_mean:.4f}')


---


In [ ]:
# ============================================================
# §4. INFORMATION CRITERIA - AIC, BIC, Delta-chi2
# ============================================================
# k_EU = k_LCDM = 7 (6 cosmo + A_planck)
# EU params (eps_IR, z_trans, b) are QFT-fixed, NOT fitted.
# -> Delta AIC = Delta BIC = Delta chi2
# ============================================================

print('=' * 60)
print('INFORMATION CRITERIA')
print('=' * 60)

k_eu   = 7
k_lcdm = 7

print(f'\n  Free parameters:')
print(f'    k_EU   = {k_eu} (6 cosmo + A_planck)')
print(f'    k_LCDM = {k_lcdm} (6 cosmo + A_planck)')
print(f'    EU extras: 0 (eps_IR, z_trans, b fixed by QFT)')
print(f'    -> k_EU = k_LCDM -> Delta_AIC = Delta_BIC = Delta_chi2')

delta_chi2_base = chi2_eu_base - chi2_lcdm_base
delta_AIC_base = delta_chi2_base
delta_BIC_base = delta_chi2_base

delta_chi2_ext = chi2_eu_ext - chi2_lcdm_ext
delta_AIC_ext = delta_chi2_ext
delta_BIC_ext = delta_chi2_ext

def interpret(delta):
    d = abs(delta)
    if d < 2:   return 'Indistinguishable'
    elif d < 6: return 'Positive'
    elif d < 10: return 'Strong'
    else:       return 'DECISIVE'

def favor(delta):
    if delta < -0.1: return 'EU'
    elif delta > 0.1: return 'LCDM'
    else: return 'Tie'

print(f'\n{"="*70}')
print(f'{"GRAND TABLE - EU vs LCDM Model Selection":^70}')
print(f'{"="*70}')
print(f'{"Dataset":<25} {"chi2_EU":>10} {"chi2_LCDM":>10} {"D_AIC":>8} {"Evidence":>18} {"Favors":>8}')
print(f'{"-"*70}')
print(f'{"CMB+BAO+SNe":<25} {chi2_eu_base:10.1f} {chi2_lcdm_base:10.1f} {delta_AIC_base:+8.1f} {interpret(delta_AIC_base):>18} {favor(delta_AIC_base):>8}')
print(f'{"+SH0ES+DES":<25} {chi2_eu_ext:10.1f} {chi2_lcdm_ext:10.1f} {delta_AIC_ext:+8.1f} {interpret(delta_AIC_ext):>18} {favor(delta_AIC_ext):>8}')
print(f'{"-"*70}')
print(f'\n  Note: Delta_AIC < 0 -> EU is favored. |Delta_AIC| > 10 -> decisive.')

---


In [ ]:
# ============================================================
# §5. CHI2 BREAKDOWN - By likelihood
# ============================================================

print('=' * 60)
print('CHI2 BREAKDOWN BY LIKELIHOOD')
print('=' * 60)

# EU chi2 breakdown from D2
eu_chi2 = d2['chi2_bestfit']

# LCDM breakdown: from cache or from live minimizer
if USE_CACHE:
    # lcdm_chi2 already loaded in §3 from _cache['extended']['per_likelihood']
    pass
else:
    # lcdm_chi2 already built in §3 from bf_ext.columns
    pass
# Either way, lcdm_chi2 is a dict {key: value} ready to use.

components = [
    ('Planck NPIPE TTTEEE', 'planck_NPIPE_highl_CamSpec.TTTEEE'),
    ('Planck lowl TT',      'planck_2018_lowl.TT'),
    ('Planck lowl EE',      'planck_2018_lowl.EE'),
    ('Planck lensing',      'planck_2018_lensing.clik'),
    ('DESI DR2 BAO',        'bao.desi_dr2'),
    ('Pantheon+',           'sn.pantheonplus'),
]

print(f'\n{"Likelihood":<42} {"chi2_EU":>9} {"chi2_LCDM":>9} {"Delta":>8}')
print(f'{"-"*70}')

for name, key in components:
    eu_val = eu_chi2.get(key, None)
    lcdm_val = lcdm_chi2_base.get(key, None)
    if eu_val is not None and lcdm_val is not None:
        delta = eu_val - lcdm_val
        print(f'  {name:<40} {eu_val:9.1f} {lcdm_val:9.1f} {delta:+8.1f}')
    elif eu_val is not None:
        print(f'  {name:<40} {eu_val:9.1f} {"N/A":>9} {"--":>8}')

print(f'{"-"*70}')
print(f'  {"SH0ES (Breuval 2024)":<40} {chi2_shoes_eu:9.2f} {chi2_shoes_lcdm:9.1f} {chi2_shoes_eu - chi2_shoes_lcdm:+8.1f}')
print(f'{"-"*70}')
print(f'  {"TOTAL (Extended)":<40} {chi2_eu_ext:9.1f} {chi2_lcdm_ext:9.1f} {delta_AIC_ext:+8.1f}')


---


In [ ]:
# ============================================================
# §6. TENSION SUMMARY
# ============================================================

print('=' * 60)
print('TENSION SUMMARY')
print('=' * 60)

# H0 tension: EU H0_LKI vs SH0ES (model-independent measurement)
H0_LKI_eu_base = c2['derived_params']['H0_LKI']['mean']

# S8 tension: EU internal consistency (NB03-B prediction vs D2 posterior)
# NOT compared to DES S8=0.776 (which has LCDM kernel bias, see NB08)
S8_eu_pred     = nb03b['sigma8']['S8_eu']       # CLASS-EU prediction
S8_eu_d2_mean  = d2['derived_params']['S8']['mean']
S8_eu_d2_std   = d2['derived_params']['S8']['std']

tensions = {
    'H0 (SH0ES)': {
        'LCDM': abs(H0_lcdm_bf - H0_SHOES) / H0_SHOES_ERR,
        'EU':   abs(H0_LKI_eu_base - H0_SHOES) / H0_SHOES_ERR,
    },
    'S8 (EU internal)': {
        'LCDM': abs(S8_lcdm_bf - 0.776) / 0.017,    # LCDM vs DES (standard)
        'EU':   abs(S8_eu_d2_mean - S8_eu_pred) / S8_eu_d2_std,  # EU prediction vs EU+DES fit
    },
}

print(f'\n{"Tension":<25} {"LCDM":>10} {"EU":>10} {"Reduction":>12}')
print(f'{"-"*60}')
for name, vals in tensions.items():
    reduction = vals['LCDM'] - vals['EU']
    print(f'  {name:<23} {vals["LCDM"]:8.2f}s {vals["EU"]:8.2f}s {reduction:+10.2f}s')

print(f'\n  -> EU resolves H0 from ~{tensions["H0 (SH0ES)"]["LCDM"]:.0f}s to ~{tensions["H0 (SH0ES)"]["EU"]:.1f}s')
print(f'  -> S8 internal consistency: {tensions["S8 (EU internal)"]["EU"]:.2f}s')
print(f'     (NB03-B pred: {S8_eu_pred:.4f} vs D2+DES: {S8_eu_d2_mean:.4f} +/- {S8_eu_d2_std:.4f})')
print(f'  -> LCDM S8 vs DES: {tensions["S8 (EU internal)"]["LCDM"]:.1f}s (standard comparison)')
print(f'  -> Note: DES S8=0.776 has LCDM kernel bias. See NB08 for full analysis.')


---


In [ ]:
# ============================================================
# §7. PLOTS - Model Selection
# ============================================================
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.family': 'serif', 'font.size': 12, 'axes.labelsize': 14, 'figure.facecolor': 'white'})
os.makedirs('figures', exist_ok=True)

# -- Plot 1: Delta AIC bar chart --
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
datasets = ['Baseline\n(CMB+BAO+SNe)', 'Extended\n(+SH0ES+DES)']
delta_vals = [delta_AIC_base, delta_AIC_ext]
colors = ['#4A90D9' if d > -2 else '#E74C3C' for d in delta_vals]

bars = ax.bar(datasets, delta_vals, color=colors, width=0.5, edgecolor='black', linewidth=0.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(-10, color='red', linewidth=0.8, linestyle='--', alpha=0.5, label='|DAIC| = 10 (decisive)')
ax.axhline(10, color='red', linewidth=0.8, linestyle='--', alpha=0.5)

for bar, val in zip(bars, delta_vals):
    y = val - 2 if val < 0 else val + 1
    ax.text(bar.get_x() + bar.get_width()/2, y, f'{val:+.1f}',
            ha='center', va='top' if val < 0 else 'bottom', fontweight='bold', fontsize=13)

ax.set_ylabel('DAIC  (EU - LCDM)')
ax.set_title('Model Selection: EU vs LCDM\n(DAIC < 0 = EU favored, |DAIC| > 10 = decisive)')
ax.legend(loc='upper right', fontsize=10)
ax.set_ylim(min(delta_vals) * 1.3, max(10, max(delta_vals) * 1.3))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('figures/fig_NB06_DAIC.png', dpi=150, bbox_inches='tight')
plt.savefig('figures/fig_NB06_DAIC.pdf', bbox_inches='tight')
plt.show()
print('[OK] fig_NB06_DAIC saved')

# -- Plot 2: Chi2 decomposition --
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
labels = ['Planck\nTTTEEE', 'lowl\nTT', 'lowl\nEE', 'Lensing', 'DESI\nDR2', 'Pantheon+', 'SH0ES']
eu_vals = [
    eu_chi2.get('planck_NPIPE_highl_CamSpec.TTTEEE', 0),
    eu_chi2.get('planck_2018_lowl.TT', 0),
    eu_chi2.get('planck_2018_lowl.EE', 0),
    eu_chi2.get('planck_2018_lensing.clik', 0),
    eu_chi2.get('bao.desi_dr2', 0),
    eu_chi2.get('sn.pantheonplus', 0),
    chi2_shoes_eu,
]
lcdm_plot = [lcdm_chi2_base.get(k, 0) or 0 for _, k in components] + [chi2_shoes_lcdm]

x = np.arange(len(labels))
w = 0.35
ax.bar(x - w/2, eu_vals, w, label='EU', color='#2ECC71', edgecolor='black', linewidth=0.5)
ax.bar(x + w/2, lcdm_plot, w, label='LCDM', color='#E74C3C', edgecolor='black', linewidth=0.5)
for i in [6]:
    ax.axvspan(i - 0.5, i + 0.5, alpha=0.1, color='red')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('chi2')
ax.set_title('Chi2 Decomposition - EU vs LCDM')
ax.legend(fontsize=12)
ax.set_yscale('symlog', linthresh=50)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('figures/fig_NB06_chi2_decomposition.png', dpi=150, bbox_inches='tight')
plt.savefig('figures/fig_NB06_chi2_decomposition.pdf', bbox_inches='tight')
plt.show()
print('[OK] fig_NB06_chi2_decomposition saved')

---


In [ ]:
# ============================================================
# §8. ROBUSTNESS - Sensitivity to H0 prior
# ============================================================

print('=' * 60)
print('ROBUSTNESS - H0 prior sensitivity')
print('=' * 60)

h0_priors = [
    ('SH0ES (Breuval+ 2024)',    73.17, 0.86),
    ('TRGB (Freedman+ 2024)',    69.85, 1.75),
    ('JAGB (Freedman+ 2024)',    67.96, 1.85),
    ('Miras (Huang+ 2024)',      73.01, 1.04),
]

print(f'\n{"Prior":<30} {"H0":>6} {"err":>5} {"dchi2_LCDM":>10} {"dchi2_EU":>8} {"DDchi2":>8}')
print(f'{"-"*68}')
for name, h0, err in h0_priors:
    dchi2_lcdm = ((H0_lcdm_bf - h0) / err)**2
    dchi2_eu   = ((H0_LKI_eu_base - h0) / err)**2
    ddchi2 = dchi2_eu - dchi2_lcdm
    print(f'  {name:<28} {h0:6.2f} {err:5.2f} {dchi2_lcdm:10.1f} {dchi2_eu:8.2f} {ddchi2:+8.1f}')

print(f'\n  -> EU is favored with ALL local H0 calibrations.')

---


In [ ]:
# ============================================================
# §9. EXPORT - NB06 results
# ============================================================

print('=' * 60)
print('EXPORT - NB06 Results')
print('=' * 60)

nb06_results = {
    'metadata': {
        'notebook': 'NB06_Model_Selection',
        'version': 'v2.0_shoes_only',
        'date': str(np.datetime64('now')),
        'description': 'EU vs LCDM model selection via AIC/BIC/Dchi2',
        'upstream': ['NB05_C2_results.json', 'NB05_D2_results.json', 'NB03_B_results.json'],
        'method': 'Cobaya minimize (BOBYQA, best_of=4)',
        'used_cache': USE_CACHE,
        'k_EU': k_eu, 'k_LCDM': k_lcdm,
    },
    'baseline': {
        'datasets': 'Planck NPIPE + DESI DR2 + Pantheon+',
        'chi2_EU': float(chi2_eu_base), 'chi2_LCDM': float(chi2_lcdm_base),
        'delta_AIC': float(delta_AIC_base), 'evidence': interpret(delta_AIC_base),
        'favors': favor(delta_AIC_base),
    },
    'extended': {
        'datasets': 'Baseline + SH0ES (Breuval 2024) + DES-Y3 (proxy)',
        'chi2_EU': float(chi2_eu_ext), 'chi2_LCDM': float(chi2_lcdm_ext),
        'delta_AIC': float(delta_AIC_ext), 'evidence': interpret(delta_AIC_ext),
        'favors': favor(delta_AIC_ext),
        'chi2_shoes_EU': float(chi2_shoes_eu), 'chi2_shoes_LCDM': float(chi2_shoes_lcdm),
    },
    'lcdm_bestfit': {
        'H0': float(H0_lcdm_bf), 'sigma8': float(sigma8_lcdm_bf),
        'S8': float(S8_lcdm_bf), 'Omega_m': float(Omega_m_lcdm_bf),
        'rdrag': float(rdrag_lcdm_bf), 'chi2_total': float(chi2_lcdm_base),
    },
    'eu_reference': {
        'H0_GKI': float(c2['derived_params']['H0']['mean']),
        'H0_LKI': float(c2['derived_params']['H0_LKI']['mean']),
        'S8_modeB': float(c2['derived_params']['S8']['mean']),
        'chi2_total': float(chi2_eu_base),
    },
    'tensions': {k: v for k, v in tensions.items()},
    'eu_extended': {
        'H0_LKI_ext': float(d2['derived_params']['H0_LKI']['mean']),
        'S8_ext': float(d2['derived_params']['S8']['mean']),
        'chi2_base_d2': float(d2['chi2_bestfit']['total']),
        'd2_status': d2['_metadata']['convergence']['status'],
        'S8_eu_prediction': float(nb03b['sigma8']['S8_eu']),
        'S8_internal_tension_sigma': float(abs(S8_eu_ext - nb03b['sigma8']['S8_eu']) / d2['derived_params']['S8']['std']),
    },
}

with open('NB06_results.json', 'w') as f:
    json.dump(nb06_results, f, indent=2, default=float)
print(f'  [SAVED] NB06_results.json')

# -- Save minimizer cache for re-use (only when BOBYQA ran) --
if not USE_CACHE:
    minimizer_cache = {
        'metadata': {
            'method': 'Cobaya minimize (BOBYQA, best_of=4)',
            'date': str(np.datetime64('now')),
            'cobaya_version': cobaya.__version__,
            'note': 'LCDM best-fit only. Model-independent. Reusable across D2 updates.',
        },
        'baseline': {
            'chi2_total': float(chi2_lcdm_base),
            'H0': float(H0_lcdm_bf),
            'sigma8': float(sigma8_lcdm_bf),
            'S8': float(S8_lcdm_bf),
            'Omega_m': float(Omega_m_lcdm_bf),
            'rdrag': float(rdrag_lcdm_bf),
            'per_likelihood': lcdm_chi2_base,
        },
        'extended': {
            'chi2_total': float(chi2_lcdm_ext),
            'H0': float(H0_lcdm_ext),
            'S8': float(S8_lcdm_ext),
            'chi2_shoes': float(chi2_shoes_lcdm),
            'chi2_des': float(chi2_des_lcdm),
            'per_likelihood': lcdm_chi2,
        },
    }
    with open('NB06_lcdm_minimizer.json', 'w') as f:
        json.dump(minimizer_cache, f, indent=2)
    print(f'  [SAVED] NB06_lcdm_minimizer.json (reusable LCDM cache)')
else:
    print(f'  [SKIP] NB06_lcdm_minimizer.json (used existing cache)')


---


In [ ]:
# ============================================================
# §10. COLAB DOWNLOAD
# ============================================================
import zipfile, glob

_base = '/content'
zip_path = os.path.join(_base, 'NB06_outputs.zip')
_count = 0

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Results JSON
    for jpath in [os.path.join(_base, 'NB06_results.json'), 'NB06_results.json']:
        if os.path.exists(jpath):
            zf.write(jpath, 'NB06_results.json')
            print(f'  Added: NB06_results.json')
            _count += 1
            break
    # Minimizer cache
    for jpath in [os.path.join(_base, 'NB06_lcdm_minimizer.json'), 'NB06_lcdm_minimizer.json']:
        if os.path.exists(jpath):
            zf.write(jpath, 'NB06_lcdm_minimizer.json')
            print(f'  Added: NB06_lcdm_minimizer.json')
            _count += 1
            break
    # Figures
    for _fdir in [os.path.join(_base, 'figures'), 'figures']:
        if os.path.isdir(_fdir):
            for f in sorted(glob.glob(os.path.join(_fdir, 'fig_NB06_*'))):
                arcname = os.path.join('figures', os.path.basename(f))
                zf.write(f, arcname)
                print(f'  Added: {arcname}')
                _count += 1
            break

assert _count > 0, f'FATAL: No files added to zip! CWD = {os.getcwd()}'
print(f'\n[OK] NB06_outputs.zip - {_count} files ({os.path.getsize(zip_path):,} bytes)')

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print(f'Local env - file at {zip_path}')


---

## References

1. **Breuval, Riess et al. (2024)** — H₀ = 73.17 ± 0.86 km/s/Mpc
2. **DES-Y3** — S₈ = 0.776 ± 0.017 (cosmic shear)
3. **Planck NPIPE** — CamSpec TTTEEE + lowl TT/EE + lensing
4. **DESI DR2** — BAO measurements (2024)
5. **Pantheon+** — Type Ia supernovae (Brout+ 2022)
6. **Cobaya** — Torrado & Lewis (2021)
7. **Burnham & Anderson (2002)** — AIC/BIC interpretation scale
